# Stages 1-4 End-to-End Test Runner

Walks a single query through flow routing (stage 1), query understanding (stage 2), stage-3 orchestration (translate + execute every endpoint with full visibility), and stage-4 reranking (pool scoring and top-K ranking). Unlike `run_stage_4`, this notebook captures every stage-3 LLM spec so they can be inspected alongside the per-endpoint execution results.

## Cell 1 — Setup, imports, DB readiness

Run once. Opens the Postgres pool, initializes Redis, and pings Qdrant (idempotent — safe to rerun). Every import used anywhere below lives here.

In [7]:
import sys, os, time, asyncio, json
from datetime import date, datetime, timezone
from pathlib import Path
from typing import Iterable, Protocol

# Ensure project root is on the path so imports resolve.
project_root = str(Path.cwd().parent) if Path.cwd().name == "search_v2" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dotenv import load_dotenv
load_dotenv(Path(project_root) / ".env")

from implementation.llms.generic_methods import LLMProvider

# Stage 1 / 2
from search_v2.stage_1 import route_query
from search_v2.stage_2 import understand_query

# Stage-4 building blocks (also drive stage 3 end-to-end with spec capture)
from search_v2.stage_4.flow_detection import detect_flow, tag_items
from search_v2.stage_4.dispatch import translate_item, execute_item
from search_v2.stage_4.assembly import assemble_pool, apply_deterministic_exclusions
from search_v2.stage_4.scoring import score_pool
from search_v2.stage_4.display import build_display_payload
from search_v2.stage_4.types import EndpointOutcome, Stage4Flow

# Shared schemas
from schemas.enums import EndpointRoute, SearchFlow
from schemas.endpoint_result import EndpointResult
from schemas.movie_input import load_movie_input_data

# DB
from db.postgres import pool as postgres_pool, fetch_movie_cards
from db.qdrant import qdrant_client, check_qdrant
from db.redis import init_redis, check_redis
import db.redis as _redis_module

# DB readiness (idempotent — mirrors test_stage_3.ipynb)
if postgres_pool._closed:
    await postgres_pool.open()
    print("Postgres: pool opened")
else:
    print("Postgres: pool already open")

if _redis_module._redis_client is None:
    await init_redis()
    print(f"Redis:    initialized ({await check_redis()})")
else:
    print(f"Redis:    already initialized ({await check_redis()})")

print(f"Qdrant:   {await check_qdrant()}")

Postgres: pool already open
Redis:    already initialized (ok)
Qdrant:   ok


## Cell 2 — Configuration

Set the query, LLM preset, and top-K. Uncomment one preset block or assign the three variables manually.

In [8]:
# ---- Query ----
query = "Family friendly Disney live action from the 2000s"

# ---- Presets (uncomment one) ----
# Kimi K2.5 — no thinking
# provider, model, kwargs = LLMProvider.KIMI, "kimi-k2.5", {"enable_thinking": False}

# GPT 5.4 Mini — no thinking
provider, model, kwargs = LLMProvider.OPENAI, "gpt-5.4-mini", {"reasoning_effort": "none", "verbosity": "low"}

# Gemini 3 Flash — no thinking
# provider, model, kwargs = LLMProvider.GEMINI, "gemini-3-flash-preview", {"thinking_config": {"thinking_budget": 0}}

# ---- Shared ----
today = date.today()
TOP_K = 100

print(f"Query:    {query}")
print(f"Provider: {provider.value}")
print(f"Model:    {model}")
print(f"Kwargs:   {kwargs or '(defaults)'}")
print(f"Today:    {today.isoformat()}")
print(f"Top-K:    {TOP_K}")

Query:    Family friendly Disney live action from the 2000s
Provider: openai
Model:    gpt-5.4-mini
Kwargs:   {'reasoning_effort': 'none', 'verbosity': 'low'}
Today:    2026-04-21
Top-K:    100


## Cell 3 — Stage 1: Flow Routing

In [9]:
start = time.perf_counter()
stage_1_response, stage_1_in_tok, stage_1_out_tok = await route_query(query)
stage_1_elapsed = time.perf_counter() - start

print(f"Stage 1 completed in {stage_1_elapsed:.2f}s")
print(f"Tokens — input: {stage_1_in_tok:,}  output: {stage_1_out_tok:,}\n")
print(stage_1_response.model_dump_json(indent=2))

Stage 1 completed in 1.79s
Tokens — input: 1,977  output: 154

{
  "ambiguity_analysis": "The query is clear but broad, specifying a brand, format, and decade without a specific title or sub-genre preference.",
  "ambiguity_level": "clear",
  "hard_constraints": [
    "Disney",
    "live action",
    "2000s",
    "family friendly"
  ],
  "ambiguity_sources": [],
  "primary_intent": {
    "routing_signals": "The query specifies 'Disney', 'live action', '2000s', and 'Family friendly' as metadata filters.",
    "intent_rewrite": "Find Disney live-action family movies released between 2000 and 2009.",
    "flow": "standard",
    "display_phrase": "Disney Live Action from the 2000s",
    "title": null
  },
  "alternative_intents": []
}


## Cell 4 — Stage 2: Query Understanding

Picks the first STANDARD-flow intent from stage 1 and runs query decomposition on its `intent_rewrite`. Non-standard flows (title lookup, similarity) aren't supported by stages 3-4 in this notebook.

In [10]:
# Pick the first standard-flow intent. Change this filter or index to test a different one.
stage_1_intents = [stage_1_response.primary_intent, *stage_1_response.alternative_intents]
selected_intent = next(
    (intent for intent in stage_1_intents if intent.flow == SearchFlow.STANDARD),
    None,
)
if selected_intent is None:
    raise RuntimeError(
        "No standard-flow intent in stage 1 output — this notebook "
        "only drives the standard pipeline. Tweak the query."
    )

intent_rewrite = selected_intent.intent_rewrite
print(f"Selected intent: {selected_intent.display_phrase}")
print(f"Intent rewrite:          {intent_rewrite}\n")

start = time.perf_counter()
qu, stage_2_in_tok, stage_2_out_tok = await understand_query(
    query=intent_rewrite, provider=provider, model=model, **kwargs
)
stage_2_elapsed = time.perf_counter() - start

print(f"Stage 2 completed in {stage_2_elapsed:.2f}s")
print(f"Tokens — input: {stage_2_in_tok:,}  output: {stage_2_out_tok:,}\n")
print(qu.model_dump_json(indent=2))

Selected intent: Disney Live Action from the 2000s
Intent rewrite:          Find Disney live-action family movies released between 2000 and 2009.

Stage 2 completed in 2.28s
Tokens — input: 9,248  output: 215

{
  "decomposition_analysis": "The query asks for Disney-produced movies, so the studio is a hard requirement. It also requires live-action, family classification, and a 2000s release window, all of which are hard filtering constraints rather than ranking qualities.",
  "dealbreakers": [
    {
      "description": "includes Disney as a production company",
      "direction": "inclusion",
      "routing_rationale": "studio / production company",
      "route": "studio"
    },
    {
      "description": "is live action",
      "direction": "inclusion",
      "routing_rationale": "keyword form-factor: live action",
      "route": "keyword"
    },
    {
      "description": "is a family movie",
      "direction": "inclusion",
      "routing_rationale": "keyword family: family",
     

## Cell 5 — Stage 3 Orchestration

Translates every tagged item, executes the pool-independent ones unrestricted, then assembles the candidate pool and applies deterministic exclusions. Prints the LLM spec and execution result for each endpoint, followed by a consolidated stats block.

Pool-dependent items (inclusion dealbreakers / preferences that don't generate candidates, plus semantic exclusions) have their LLM specs printed here but their execution is deferred to stage 4 since it needs the assembled pool.

In [11]:
class _Scored(Protocol):
    movie_id: int
    score: float


async def print_scored_candidates(
    candidates: Iterable[_Scored],
    *,
    limit: int | None = 10,
    sort_desc: bool = True,
) -> None:
    """Print scored movies as `score  title (year)  [tmdb_id]`."""
    items_list = list(candidates)
    if not items_list:
        print("    (no scored candidates)")
        return
    if sort_desc:
        items_list = sorted(items_list, key=lambda c: c.score, reverse=True)
    if limit is not None:
        items_list = items_list[:limit]
    cards = await fetch_movie_cards([c.movie_id for c in items_list])
    by_id = {c["movie_id"]: c for c in cards}
    for cand in items_list:
        card = by_id.get(cand.movie_id)
        if card is None:
            title, year = "<missing card>", "?"
        else:
            title = card["title"] or "<untitled>"
            ts = card["release_ts"]
            year = (
                datetime.fromtimestamp(ts, tz=timezone.utc).year
                if ts is not None else "?"
            )
        print(f"    {cand.score:.3f}  {title} ({year})  [tmdb_id={cand.movie_id}]")


# Phase 0 — flow detect + item tagging -------------------------------------
flow = detect_flow(qu)
items = tag_items(qu, flow)

print(f"Stage-4 flow: {flow.value}")
if flow == Stage4Flow.BROWSE:
    raise RuntimeError(
        "BROWSE flow not supported by this notebook — the seed would come "
        "from movie_card priors; extend with fetch_browse_seed_prior_inputs."
    )

print(f"Tagged items ({len(items)}):")
for it in items:
    print(
        f"  {it.debug_key:32s}  endpoint={it.endpoint.value:22s}  "
        f"role={it.role:22s}  generates_candidates={it.generates_candidates}"
    )

# Partition items the same way run_stage_4 does.
candidate_generators = [i for i in items if i.generates_candidates]
det_exclusions = [
    i for i in items
    if i.role == "exclusion_dealbreaker" and i.endpoint != EndpointRoute.SEMANTIC
]
sem_exclusions = [
    i for i in items
    if i.role == "exclusion_dealbreaker" and i.endpoint == EndpointRoute.SEMANTIC
]
pool_dependent_items = (
    [i for i in items if i.role == "inclusion_dealbreaker" and not i.generates_candidates]
    + [i for i in items if i.role == "preference" and not i.generates_candidates]
    + sem_exclusions
)
pool_independent_items = candidate_generators + det_exclusions

# Phase 1 — translate every item in parallel (captures specs for stage 4).
translate_kwargs = dict(
    intent_rewrite=intent_rewrite,
    today=today,
    provider=provider,
    model=model,
)

start = time.perf_counter()
translate_results = await asyncio.gather(
    *(translate_item(i, **translate_kwargs) for i in items)
)
translate_elapsed = time.perf_counter() - start

specs_by_key: dict[str, tuple] = {
    it.debug_key: res for it, res in zip(items, translate_results)
}
print(f"\nAll {len(items)} translations completed in {translate_elapsed:.2f}s")

# Phase 2 — execute pool-independent items unrestricted in parallel.
start = time.perf_counter()
execute_results = await asyncio.gather(
    *(
        execute_item(it, specs_by_key[it.debug_key][0], None, qdrant_client=qdrant_client)
        for it in pool_independent_items
    )
)
execute_elapsed = time.perf_counter() - start

pool_independent_outcomes: list[EndpointOutcome] = []
for it, (result, exec_ms, exec_status, exec_err) in zip(pool_independent_items, execute_results):
    spec, llm_ms, translate_status, translate_err = specs_by_key[it.debug_key]
    # Mirror run_stage_4: if translation failed, surface the translation status.
    if translate_status != "ok":
        pool_independent_outcomes.append(EndpointOutcome(
            item=it, result=EndpointResult(), status=translate_status,
            llm_ms=llm_ms, exec_ms=None, error_message=translate_err,
        ))
    else:
        pool_independent_outcomes.append(EndpointOutcome(
            item=it, result=result, status=exec_status,
            llm_ms=llm_ms, exec_ms=exec_ms, error_message=exec_err,
        ))

print(
    f"All {len(pool_independent_items)} pool-independent executions "
    f"completed in {execute_elapsed:.2f}s\n"
)

# Per-endpoint printing.
print("=" * 72)
print("STAGE 3 PER-ENDPOINT OUTPUT")
print("=" * 72)

outcomes_by_key = {o.item.debug_key: o for o in pool_independent_outcomes}
pool_independent_keys = {it.debug_key for it in pool_independent_items}

for it in items:
    spec, llm_ms, translate_status, translate_err = specs_by_key[it.debug_key]
    print()
    print("-" * 72)
    header_llm = f"{llm_ms:.0f}ms" if llm_ms is not None else "n/a"
    print(
        f"[{it.debug_key}]  endpoint={it.endpoint.value}  role={it.role}  "
        f"LLM={header_llm}  translate_status={translate_status}"
    )
    print("-" * 72)
    print("LLM SPEC:")
    if translate_status != "ok":
        print(f"  translation {translate_status}: {translate_err}")
    elif spec is None:
        print("  (trending: no LLM translator)")
    else:
        print(spec.model_dump_json(indent=2))

    if it.debug_key in pool_independent_keys:
        outcome = outcomes_by_key[it.debug_key]
        ms = f"{outcome.exec_ms:.0f}ms" if outcome.exec_ms is not None else "n/a"
        print(
            f"\nEXECUTION — {len(outcome.result.scores)} matches  "
            f"exec={ms}  status={outcome.status}"
        )
        await print_scored_candidates(outcome.result.scores, limit=10)
    else:
        print("\nEXECUTION — deferred to stage 4 (pool-dependent)")

# Phases 3 + 4 — pool assembly + deterministic exclusion subtraction.
candidate_outcomes = [o for o in pool_independent_outcomes if o.item.generates_candidates]
det_exclusion_outcomes = [
    o for o in pool_independent_outcomes if o.item.role == "exclusion_dealbreaker"
]

pool = assemble_pool(candidate_outcomes, browse_seed_ids=None)
pool_after_generation = len(pool)
pool = apply_deterministic_exclusions(pool, det_exclusion_outcomes)
pool_after_exclusion = len(pool)

# Consolidated stats.
print()
print("=" * 72)
print("STAGE 3 CONSOLIDATED CANDIDATE STATS")
print("=" * 72)

print(f"\nCandidate generators ({len(candidate_outcomes)}):")
for o in candidate_outcomes:
    print(
        f"  {o.item.debug_key:32s}  endpoint={o.item.endpoint.value:22s}  "
        f"role={o.item.role:22s}  matched={len(o.result.scores)}"
    )

print(f"\nDeterministic exclusions ({len(det_exclusion_outcomes)}):")
for o in det_exclusion_outcomes:
    print(
        f"  {o.item.debug_key:32s}  endpoint={o.item.endpoint.value:22s}  "
        f"excluded_ids={len(o.result.scores)}"
    )

print()
print(f"Unioned inclusion IDs (pre-exclusions):  {pool_after_generation}")
print(f"Final candidate pool size:               {pool_after_exclusion}")

Stage-4 flow: standard
Tagged items (4):
  inclusion_dealbreaker[0]          endpoint=studio                  role=inclusion_dealbreaker   generates_candidates=True
  inclusion_dealbreaker[1]          endpoint=keyword                 role=inclusion_dealbreaker   generates_candidates=True
  inclusion_dealbreaker[2]          endpoint=keyword                 role=inclusion_dealbreaker   generates_candidates=True
  inclusion_dealbreaker[3]          endpoint=metadata                role=inclusion_dealbreaker   generates_candidates=True

All 4 translations completed in 2.33s
All 4 pool-independent executions completed in 3.62s

STAGE 3 PER-ENDPOINT OUTPUT

------------------------------------------------------------------------
[inclusion_dealbreaker[0]]  endpoint=studio  role=inclusion_dealbreaker  LLM=977ms  translate_status=ok
------------------------------------------------------------------------
LLM SPEC:
{
  "thinking": "This is an umbrella / parent-brand query: the registry covers Di

## Cell 6 — Stage 4 Reranking

Runs pool-dependent items against the assembled pool, fetches movie cards, scores with `score_pool`, sorts, and slices the top-K. You can set:

- `inspect_tmdb_ids` — any TMDB IDs you want a full per-endpoint breakdown for (works for any pool member, not just the top 100).
- `SHOW_OVERVIEWS` — flip to True to also print the overview text of the top-25 movies (fetched from the ingestion tracker DB).

In [12]:
# ---- User inputs ----
# TMDB IDs to drill into. Works for any id that made it through stage 3 exclusions;
# if an id was excluded at stage 3 it prints a clear message.
inspect_tmdb_ids: list[int] = []

# Flip to True to print overview text for each of the top-25 movies.
SHOW_OVERVIEWS = False

TOP_N_SHOW = 25

if not pool:
    raise RuntimeError("Pool is empty — cannot run stage 4. Check stage 3 output.")

pool_set = set(pool)

async def _run_pool_dependent(it):
    spec, llm_ms, translate_status, translate_err = specs_by_key[it.debug_key]
    if translate_status != "ok":
        return EndpointOutcome(
            item=it, result=EndpointResult(), status=translate_status,
            llm_ms=llm_ms, exec_ms=None, error_message=translate_err,
        )
    result, exec_ms, exec_status, exec_err = await execute_item(
        it, spec, pool_set, qdrant_client=qdrant_client,
    )
    return EndpointOutcome(
        item=it, result=result, status=exec_status,
        llm_ms=llm_ms, exec_ms=exec_ms, error_message=exec_err,
    )

# Pool-dependent executions + movie-card fetch run concurrently (mirrors run_stage_4 phase 5).
start = time.perf_counter()
pool_dep_coros = [_run_pool_dependent(it) for it in pool_dependent_items]
cards_task = fetch_movie_cards(pool)
pool_dependent_outcomes = await asyncio.gather(*pool_dep_coros)
cards = await cards_task
stage_4_exec_elapsed = time.perf_counter() - start
cards_by_id = {c["movie_id"]: c for c in cards}

print(
    f"Pool-dependent executions + card fetch completed in "
    f"{stage_4_exec_elapsed:.2f}s  (pool={len(pool)}, "
    f"pool-dep items={len(pool_dependent_items)})"
)

# Phase 6 — score composition.
all_outcomes = list(pool_independent_outcomes) + list(pool_dependent_outcomes)
inclusion_outcomes = [o for o in all_outcomes if o.item.role == "inclusion_dealbreaker"]
preference_outcomes = [o for o in all_outcomes if o.item.role == "preference"]
semantic_exclusion_outcomes = [
    o for o in all_outcomes
    if o.item.role == "exclusion_dealbreaker" and o.item.endpoint == EndpointRoute.SEMANTIC
]

prior_inputs = {
    mid: (
        cards_by_id.get(mid, {}).get("reception_score"),
        cards_by_id.get(mid, {}).get("popularity_score"),
    )
    for mid in pool
}

breakdowns = score_pool(
    pool,
    inclusion_outcomes=inclusion_outcomes,
    preference_outcomes=preference_outcomes,
    semantic_exclusion_outcomes=semantic_exclusion_outcomes,
    prior_inputs=prior_inputs,
    quality_prior_mode=qu.quality_prior,
    notability_prior_mode=qu.notability_prior,
)

# Phase 7 — sort, slice, shape.
breakdowns_sorted = sorted(breakdowns, key=lambda b: b.final_score, reverse=True)
top = breakdowns_sorted[:TOP_K]
top_ids = [b.movie_id for b in top]
breakdown_by_id = {b.movie_id: b for b in breakdowns}
movies = build_display_payload(top_ids, cards_by_id)
top_25 = movies[:TOP_N_SHOW]

# debug_key → endpoint lookup for readability in the breakdown block.
endpoint_by_key = {it.debug_key: it.endpoint.value for it in items}

# Per-endpoint breakdowns for inspected TMDB IDs.
print()
print("=" * 72)
print("PER-ENDPOINT BREAKDOWNS FOR inspect_tmdb_ids")
print("=" * 72)

if not inspect_tmdb_ids:
    print("\n(no ids supplied — set `inspect_tmdb_ids = [...]` to drill in)")
else:
    for tmdb_id in inspect_tmdb_ids:
        print()
        card = cards_by_id.get(tmdb_id)
        title = card["title"] if card else "<card unavailable>"
        b = breakdown_by_id.get(tmdb_id)
        if b is None:
            print(f"  {tmdb_id}: {title} — not in scored pool (excluded at stage 3)")
            continue
        print(f"  {tmdb_id}: {title}")
        print(f"    final_score             = {b.final_score:.4f}")
        print(f"    dealbreaker_sum         = {b.dealbreaker_sum:.4f}")
        print(f"    preference_contribution = {b.preference_contribution:.4f}")
        print(f"    exclusion_penalties     = {b.exclusion_penalties:.4f}")
        if b.per_item_scores:
            print("    per-endpoint contributions:")
            for debug_key, score in b.per_item_scores.items():
                ep = endpoint_by_key.get(debug_key, "prior")
                print(f"      {debug_key:32s}  endpoint={ep:22s}  score={score:.4f}")
        else:
            print("    (no endpoint contributed a non-zero score)")

# First 25 of top 100.
print()
print("=" * 72)
print(f"TOP {TOP_N_SHOW} OF TOP {TOP_K}")
print("=" * 72)
for rank, (movie, b) in enumerate(zip(top_25, top[:TOP_N_SHOW]), start=1):
    title = movie["movie_title"] or "<untitled>"
    release_date = movie["release_date"] or ""
    year = release_date[:4] if release_date else "?"
    print(
        f"  #{rank:<3d}  score={b.final_score:.4f}  "
        f"{title} ({year})  tmdb_id={movie['tmdb_id']}"
    )

# Optional overviews.
if SHOW_OVERVIEWS and top_25:
    print()
    print("=" * 72)
    print(f"OVERVIEWS FOR TOP {TOP_N_SHOW}")
    print("=" * 72)
    overview_data = load_movie_input_data([m["tmdb_id"] for m in top_25])
    for rank, movie in enumerate(top_25, start=1):
        tmdb_id = movie["tmdb_id"]
        title = movie["movie_title"] or "<untitled>"
        data = overview_data.get(tmdb_id)
        overview = (data.overview if data else "") or "(overview unavailable)"
        print()
        print(f"  #{rank}  {title}  [tmdb_id={tmdb_id}]")
        print(f"  {overview}")

Pool-dependent executions + card fetch completed in 8.92s  (pool=107998, pool-dep items=0)

PER-ENDPOINT BREAKDOWNS FOR inspect_tmdb_ids

(no ids supplied — set `inspect_tmdb_ids = [...]` to drill in)

TOP 25 OF TOP 100
  #1    score=4.7205  Bridge to Terabithia (2007)  tmdb_id=1265
  #2    score=4.7168  The Chronicles of Narnia: The Lion, the Witch and the Wardrobe (2005)  tmdb_id=411
  #3    score=4.7003  Finding Neverland (2004)  tmdb_id=866
  #4    score=4.6989  Holes (2003)  tmdb_id=8326
  #5    score=4.6960  The Rookie (2002)  tmdb_id=14635
  #6    score=4.6763  Freaky Friday (2003)  tmdb_id=10330
  #7    score=4.6739  Eight Below (2006)  tmdb_id=9036
  #8    score=4.6553  Spy Kids (2001)  tmdb_id=10054
  #9    score=4.6553  Tuck Everlasting (2002)  tmdb_id=13768
  #10   score=4.6421  The Chronicles of Narnia: Prince Caspian (2008)  tmdb_id=2454
  #11   score=4.6340  Sky High (2005)  tmdb_id=11459
  #12   score=4.6283  Ghosts of the Abyss (2003)  tmdb_id=24982
  #13   score=4.620